<a href="https://colab.research.google.com/github/mukta121/preprosessing/blob/main/chunking_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import json
from pathlib import Path


# ============================================================
# 1. CONFIGURATION
# ============================================================

BASE_DIR = Path(
    r"C:\Users\mm7453\OneDrive - Health\workplace\adhoc\SOBs"
)

SOURCE_DIR = BASE_DIR / "cleaned_md_files"

OUTPUT_DIR = BASE_DIR / "chunked_files"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_JSONL = OUTPUT_DIR / "sob_chunks.jsonl"


# Approximate chunk size
# 3200 chars is roughly 700-900 tokens for English text.
MAX_CHARS = 3200

# Used only when a long text section must be split.
OVERLAP_CHARS = 400

# Avoid very tiny chunks.
MIN_CHUNK_CHARS = 100


# ============================================================
# 2. NORMALIZE TEXT
# ============================================================

def normalize_text(text):
    """
    Normalize line endings and excessive blank lines
    without destroying Markdown structure.
    """

    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Remove trailing whitespace
    text = re.sub(
        r"[ \t]+$",
        "",
        text,
        flags=re.MULTILINE
    )

    # Reduce excessive blank lines
    text = re.sub(
        r"\n\s*\n\s*\n+",
        "\n\n",
        text
    )

    return text.strip()


# ============================================================
# 3. DOCUMENT ID
# ============================================================

def get_document_id(file_path):
    """
    Example:

    DN0000000113.md
        ->
    DN0000000113
    """

    return file_path.stem


# ============================================================
# 4. DETECT PRODUCT TYPE
# ============================================================

def detect_product_type(plan_name):
    """
    Infer product type from plan name.

    Examples:
        BEST BUY HMO - LP -> HMO
        Some PPO Plan     -> PPO
    """

    if not plan_name:
        return None

    name = plan_name.upper()

    product_patterns = [
        ("HMO", "HMO"),
        ("PPO", "PPO"),
        ("EPO", "EPO"),
        ("POS", "POS"),
        ("HSA", "HSA"),
        ("HDHP", "HDHP"),
    ]

    for keyword, product_type in product_patterns:

        if keyword in name:
            return product_type

    return None


# ============================================================
# 5. EXTRACT DOCUMENT METADATA
# ============================================================

def extract_document_metadata(text, file_path):
    """
    Extract useful SOB metadata.

    Metadata:
        document_id
        source_file
        plan_name
        state
        effective_date
        form_number
        product_type
    """

    metadata = {
        "document_id": get_document_id(file_path),
        "source_file": file_path.name,
        "plan_name": None,
        "state": None,
        "effective_date": None,
        "form_number": None,
        "product_type": None,
    }


    # --------------------------------------------------------
    # EFFECTIVE DATE
    #
    # Examples:
    # EFFECTIVE DATE: 01/01/2026
    # PageFooter="EFFECTIVE DATE: 01/01/2026"
    # --------------------------------------------------------

    match = re.search(
        r"EFFECTIVE\s+DATE\s*:\s*"
        r"(\d{1,2}/\d{1,2}/\d{4})",
        text,
        flags=re.IGNORECASE
    )

    if match:

        metadata["effective_date"] = (
            match.group(1).strip()
        )


    # --------------------------------------------------------
    # FORM NUMBER
    #
    # Examples:
    # FORM #1574_16
    # PageFooter="FORM #1574_16"
    # --------------------------------------------------------

    match = re.search(
        r"FORM\s*#\s*([A-Za-z0-9_\-\.]+)",
        text,
        flags=re.IGNORECASE
    )

    if match:

        metadata["form_number"] = (
            match.group(1).strip()
        )


    # --------------------------------------------------------
    # FRONT MATTER
    # --------------------------------------------------------

    first_lines = []

    for line in text.splitlines()[:80]:

        clean_line = line.strip()

        if not clean_line:
            continue

        # Ignore Markdown comments
        if clean_line.startswith("<!--"):
            continue

        # Ignore obvious document wrapper heading
        if clean_line.upper() in [
            "# DOCUMENT TEXT",
            "DOCUMENT TEXT",
            "# EXTRACTED TABLES",
        ]:
            continue

        first_lines.append(
            clean_line
        )


    # --------------------------------------------------------
    # PLAN NAME DETECTION
    #
    # Example:
    # BEST BUY HMO - LP
    # --------------------------------------------------------

    plan_keywords = [
        "HMO",
        "PPO",
        "EPO",
        "POS",
        "HSA",
        "HDHP",
    ]

    for line in first_lines:

        upper_line = line.upper()

        if any(
            keyword in upper_line
            for keyword in plan_keywords
        ):

            # Avoid accidentally selecting a whole paragraph
            if len(line) <= 150:

                metadata["plan_name"] = line

                break


    # --------------------------------------------------------
    # STATE DETECTION
    # --------------------------------------------------------

    state_mapping = {
        "MAINE": "Maine",
        "MASSACHUSETTS": "Massachusetts",
        "NEW HAMPSHIRE": "New Hampshire",
        "RHODE ISLAND": "Rhode Island",
        "CONNECTICUT": "Connecticut",
        "VERMONT": "Vermont",
    }

    for line in first_lines:

        normalized = line.upper().strip()

        if normalized in state_mapping:

            metadata["state"] = (
                state_mapping[normalized]
            )

            break


    # --------------------------------------------------------
    # PRODUCT TYPE
    # --------------------------------------------------------

    metadata["product_type"] = (
        detect_product_type(
            metadata["plan_name"]
        )
    )


    return metadata


# ============================================================
# 6. PARSE MARKDOWN HEADING
# ============================================================

def parse_heading(line):
    """
    Parse Markdown headings.

    Example:

    ## Covered Benefits

    returns:

    (2, "Covered Benefits")
    """

    match = re.match(
        r"^(#{1,6})\s+(.+?)\s*$",
        line
    )

    if not match:
        return None

    level = len(
        match.group(1)
    )

    title = (
        match.group(2).strip()
    )

    return level, title


# ============================================================
# 7. PARSE DOCUMENT INTO HIERARCHICAL SECTIONS
# ============================================================

def parse_markdown_sections(text):
    """
    Convert Markdown document into sections while preserving
    heading hierarchy.

    Example:

    # DOCUMENT TEXT
    ## Covered Benefits
    ### HPHC
    #### Durable Medical Equipment

    The section hierarchy is preserved as:

    h1
    h2
    h3
    h4
    """

    sections = []

    hierarchy = {
        1: None,
        2: None,
        3: None,
        4: None,
    }

    current_section = None


    for line in text.splitlines():

        heading = parse_heading(line)

        if heading:

            level, title = heading


            # ------------------------------------------------
            # Save previous section
            # ------------------------------------------------

            if current_section:

                content = "\n".join(
                    current_section["content"]
                ).strip()

                if content:

                    current_section["content"] = (
                        content
                    )

                    sections.append(
                        current_section
                    )


            # ------------------------------------------------
            # Update hierarchy
            # ------------------------------------------------

            if level <= 4:

                hierarchy[level] = title

                # Clear lower levels
                for lower_level in range(
                    level + 1,
                    5
                ):

                    hierarchy[lower_level] = None


            current_section = {
                "level": level,
                "title": title,
                "h1": hierarchy[1],
                "h2": hierarchy[2],
                "h3": hierarchy[3],
                "h4": hierarchy[4],
                "content": [],
            }


        else:

            # ------------------------------------------------
            # Content before first heading
            # ------------------------------------------------

            if current_section is None:

                current_section = {
                    "level": 0,
                    "title": "Front Matter",
                    "h1": None,
                    "h2": None,
                    "h3": None,
                    "h4": None,
                    "content": [],
                }


            current_section["content"].append(
                line
            )


    # --------------------------------------------------------
    # Save final section
    # --------------------------------------------------------

    if current_section:

        content = "\n".join(
            current_section["content"]
        ).strip()

        if content:

            current_section["content"] = (
                content
            )

            sections.append(
                current_section
            )


    return sections


# ============================================================
# 8. BUILD SECTION PATH
# ============================================================

def build_section_path(section):
    """
    Example output:

    DOCUMENT TEXT > Covered Benefits > HPHC
    """

    parts = []

    for key in [
        "h1",
        "h2",
        "h3",
        "h4",
    ]:

        value = section.get(key)

        if value and value not in parts:

            parts.append(
                value
            )


    if not parts:

        title = section.get(
            "title"
        )

        if title:

            parts.append(
                title
            )


    return " > ".join(parts)


# ============================================================
# 9. IDENTIFY TABLE SECTIONS
# ============================================================

def is_table_section(section):
    """
    Determine whether a section belongs to
    EXTRACTED TABLES / Table 1 / Table 2 etc.
    """

    values = [
        section.get("h1"),
        section.get("h2"),
        section.get("h3"),
        section.get("h4"),
        section.get("title"),
    ]


    for value in values:

        if not value:
            continue

        value_lower = value.lower()


        if "extracted tables" in value_lower:
            return True


        if re.search(
            r"\btable\s+\d+\b",
            value_lower
        ):

            return True


    return False


# ============================================================
# 10. SPLIT LARGE NORMAL TEXT
# ============================================================

def split_text_section(
    text,
    max_chars=MAX_CHARS,
    overlap_chars=OVERLAP_CHARS
):
    """
    Split only sections that exceed MAX_CHARS.

    Paragraph boundaries are preferred.
    """

    text = text.strip()

    if len(text) <= max_chars:

        return [text]


    paragraphs = re.split(
        r"\n\s*\n",
        text
    )

    chunks = []

    current_chunk = ""


    for paragraph in paragraphs:

        paragraph = paragraph.strip()

        if not paragraph:
            continue


        # ----------------------------------------------------
        # Handle a single very large paragraph
        # ----------------------------------------------------

        if len(paragraph) > max_chars:

            if current_chunk:

                chunks.append(
                    current_chunk.strip()
                )

                current_chunk = ""


            start = 0

            while start < len(paragraph):

                end = min(
                    start + max_chars,
                    len(paragraph)
                )

                piece = paragraph[
                    start:end
                ].strip()


                if piece:

                    chunks.append(
                        piece
                    )


                if end >= len(paragraph):
                    break


                start = max(
                    end - overlap_chars,
                    start + 1
                )


            continue


        # ----------------------------------------------------
        # Try adding paragraph
        # ----------------------------------------------------

        if current_chunk:

            candidate = (
                current_chunk
                + "\n\n"
                + paragraph
            )

        else:

            candidate = paragraph


        if len(candidate) <= max_chars:

            current_chunk = candidate


        else:

            if current_chunk:

                chunks.append(
                    current_chunk.strip()
                )


            # ------------------------------------------------
            # Add controlled overlap from previous chunk
            # ------------------------------------------------

            overlap = ""

            if chunks and overlap_chars > 0:

                previous = chunks[-1]

                overlap = previous[
                    -overlap_chars:
                ]

                # Start overlap near a word boundary
                space_position = (
                    overlap.find(" ")
                )

                if space_position != -1:

                    overlap = overlap[
                        space_position + 1:
                    ]


            if overlap:

                current_chunk = (
                    overlap
                    + "\n\n"
                    + paragraph
                ).strip()

            else:

                current_chunk = (
                    paragraph
                )


    if current_chunk:

        chunks.append(
            current_chunk.strip()
        )


    return chunks


# ============================================================
# 11. SPLIT TABLE SECTION
# ============================================================

def split_table_section(
    text,
    max_chars=MAX_CHARS
):
    """
    Keep small Markdown tables intact.

    Large tables are split by rows while repeating
    the table header in every resulting chunk.
    """

    text = text.strip()


    if len(text) <= max_chars:

        return [text]


    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]


    table_lines = [
        line
        for line in lines
        if "|" in line
    ]


    # --------------------------------------------------------
    # If it doesn't look like Markdown table syntax
    # --------------------------------------------------------

    if len(table_lines) < 2:

        return split_text_section(
            text,
            max_chars=max_chars,
            overlap_chars=0
        )


    # --------------------------------------------------------
    # First 2 lines:
    #
    # | Benefit | Cost |
    # | --- | --- |
    # --------------------------------------------------------

    header_lines = (
        table_lines[:2]
    )

    data_lines = (
        table_lines[2:]
    )


    header = "\n".join(
        header_lines
    )


    chunks = []

    current_rows = []


    for row in data_lines:

        candidate_rows = (
            current_rows
            + [row]
        )

        candidate = (
            header
            + "\n"
            + "\n".join(
                candidate_rows
            )
        )


        if len(candidate) <= max_chars:

            current_rows.append(
                row
            )


        else:

            if current_rows:

                chunks.append(
                    header
                    + "\n"
                    + "\n".join(
                        current_rows
                    )
                )


            current_rows = [
                row
            ]


    if current_rows:

        chunks.append(
            header
            + "\n"
            + "\n".join(
                current_rows
            )
        )


    return chunks


# ============================================================
# 12. CREATE EMBEDDING TEXT
# ============================================================

def create_embedding_text(
    section_path,
    chunk_text,
    metadata
):
    """
    Create text that will later be passed to the embedding model.

    Important:

    Semantic metadata is included:
        plan_name
        state
        product_type
        section_path

    Exact/filter-oriented metadata is NOT repeated here:
        effective_date
        form_number
        document_id
        chunk_id

    Those remain separate metadata fields.
    """

    context = []


    if metadata.get("plan_name"):

        context.append(
            f"Plan: {metadata['plan_name']}"
        )


    if metadata.get("state"):

        context.append(
            f"State: {metadata['state']}"
        )


    if metadata.get("product_type"):

        context.append(
            f"Product Type: "
            f"{metadata['product_type']}"
        )


    if section_path:

        context.append(
            f"Section: {section_path}"
        )


    if context:

        return (
            "\n".join(context)
            + "\n\n"
            + chunk_text
        )


    return chunk_text


# ============================================================
# 13. CHUNK ONE DOCUMENT
# ============================================================

def chunk_document(file_path):
    """
    Chunk one cleaned SOB Markdown file.
    """

    text = file_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )


    text = normalize_text(
        text
    )


    document_metadata = (
        extract_document_metadata(
            text,
            file_path
        )
    )


    sections = parse_markdown_sections(
        text
    )


    chunks = []

    chunk_index = 0


    for section in sections:

        content = section[
            "content"
        ].strip()


        # Do not create tiny standalone chunks
        if len(content) < MIN_CHUNK_CHARS:
            continue


        section_path = (
            build_section_path(
                section
            )
        )


        # ----------------------------------------------------
        # TABLE CHUNKING
        # ----------------------------------------------------

        if is_table_section(section):

            chunk_type = "table"

            section_chunks = (
                split_table_section(
                    content
                )
            )


        # ----------------------------------------------------
        # TEXT CHUNKING
        # ----------------------------------------------------

        else:

            chunk_type = "text"

            section_chunks = (
                split_text_section(
                    content
                )
            )


        # ----------------------------------------------------
        # Create individual JSON records
        # ----------------------------------------------------

        for section_chunk_index, chunk_text in enumerate(
            section_chunks,
            start=1
        ):

            chunk_text = (
                chunk_text.strip()
            )


            if len(chunk_text) < MIN_CHUNK_CHARS:
                continue


            chunk_index += 1


            chunk_id = (
                f"{document_metadata['document_id']}"
                f"_chunk_{chunk_index:04d}"
            )


            embedding_text = (
                create_embedding_text(
                    section_path,
                    chunk_text,
                    document_metadata
                )
            )


            chunk_record = {

                # ===========================================
                # IDs
                # ===========================================

                "chunk_id":
                    chunk_id,

                "document_id":
                    document_metadata[
                        "document_id"
                    ],

                "source_file":
                    document_metadata[
                        "source_file"
                    ],


                # ===========================================
                # DOCUMENT METADATA
                # ===========================================

                "plan_name":
                    document_metadata.get(
                        "plan_name"
                    ),

                "state":
                    document_metadata.get(
                        "state"
                    ),

                "effective_date":
                    document_metadata.get(
                        "effective_date"
                    ),

                "form_number":
                    document_metadata.get(
                        "form_number"
                    ),

                "product_type":
                    document_metadata.get(
                        "product_type"
                    ),


                # ===========================================
                # CHUNK INFORMATION
                # ===========================================

                "chunk_index":
                    chunk_index,

                "section_chunk_index":
                    section_chunk_index,

                "chunk_type":
                    chunk_type,


                # ===========================================
                # MARKDOWN HIERARCHY
                # ===========================================

                "h1":
                    section.get(
                        "h1"
                    ),

                "h2":
                    section.get(
                        "h2"
                    ),

                "h3":
                    section.get(
                        "h3"
                    ),

                "h4":
                    section.get(
                        "h4"
                    ),

                "section_title":
                    section.get(
                        "title"
                    ),

                "section_path":
                    section_path,


                # ===========================================
                # CONTENT
                # ===========================================

                "text":
                    chunk_text,

                "embedding_text":
                    embedding_text,


                # ===========================================
                # SIZE INFORMATION
                # ===========================================

                "char_count":
                    len(
                        chunk_text
                    ),

                "approx_token_count":
                    round(
                        len(chunk_text)
                        / 4
                    ),
            }


            chunks.append(
                chunk_record
            )


    return chunks


# ============================================================
# 14. PROCESS ALL CLEANED MARKDOWN FILES
# ============================================================

def process_all_files():
    """
    Process every cleaned Markdown file and write
    all chunks to one JSONL file.
    """

    md_files = sorted(
        SOURCE_DIR.glob(
            "*.md"
        )
    )


    print(
        "=" * 75
    )

    print(
        "POINT32HEALTH SOB CHUNKING PIPELINE"
    )

    print(
        "=" * 75
    )


    print(
        f"\nSource directory:"
    )

    print(
        SOURCE_DIR
    )


    print(
        f"\nOutput JSONL:"
    )

    print(
        OUTPUT_JSONL
    )


    print(
        f"\nMarkdown files found: "
        f"{len(md_files)}"
    )


    if not md_files:

        print(
            "\nNo cleaned Markdown files found."
        )

        return


    total_chunks = 0

    total_text_chunks = 0

    total_table_chunks = 0

    successful_files = 0

    failed_files = 0


    # --------------------------------------------------------
    # Write all chunks into one JSONL file
    # --------------------------------------------------------

    with OUTPUT_JSONL.open(
        "w",
        encoding="utf-8"
    ) as jsonl_file:


        for file_path in md_files:

            try:

                print(
                    "\n" + "-" * 75
                )

                print(
                    f"Processing: "
                    f"{file_path.name}"
                )


                chunks = (
                    chunk_document(
                        file_path
                    )
                )


                # ------------------------------------------------
                # Write JSONL
                # ------------------------------------------------

                for chunk in chunks:

                    jsonl_file.write(
                        json.dumps(
                            chunk,
                            ensure_ascii=False
                        )
                        + "\n"
                    )


                # ------------------------------------------------
                # Statistics
                # ------------------------------------------------

                text_chunks = sum(
                    1
                    for chunk in chunks
                    if chunk["chunk_type"]
                    == "text"
                )


                table_chunks = sum(
                    1
                    for chunk in chunks
                    if chunk["chunk_type"]
                    == "table"
                )


                print(
                    f"    Total chunks : "
                    f"{len(chunks)}"
                )

                print(
                    f"    Text chunks  : "
                    f"{text_chunks}"
                )

                print(
                    f"    Table chunks : "
                    f"{table_chunks}"
                )


                # ------------------------------------------------
                # Print metadata found for document
                # ------------------------------------------------

                if chunks:

                    first_chunk = (
                        chunks[0]
                    )

                    print(
                        f"    Plan         : "
                        f"{first_chunk.get('plan_name')}"
                    )

                    print(
                        f"    State        : "
                        f"{first_chunk.get('state')}"
                    )

                    print(
                        f"    Effective    : "
                        f"{first_chunk.get('effective_date')}"
                    )

                    print(
                        f"    Form         : "
                        f"{first_chunk.get('form_number')}"
                    )

                    print(
                        f"    Product Type : "
                        f"{first_chunk.get('product_type')}"
                    )


                total_chunks += (
                    len(chunks)
                )

                total_text_chunks += (
                    text_chunks
                )

                total_table_chunks += (
                    table_chunks
                )

                successful_files += 1


            except Exception as e:

                failed_files += 1


                print(
                    f"\nERROR processing "
                    f"{file_path.name}"
                )

                print(
                    f"    Error type: "
                    f"{type(e).__name__}"
                )

                print(
                    f"    Error: {e}"
                )


    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    print(
        "\n" + "=" * 75
    )

    print(
        "CHUNKING COMPLETE"
    )

    print(
        "=" * 75
    )


    print(
        f"Files successful : "
        f"{successful_files}"
    )

    print(
        f"Files failed     : "
        f"{failed_files}"
    )

    print(
        f"Total chunks     : "
        f"{total_chunks}"
    )

    print(
        f"Text chunks      : "
        f"{total_text_chunks}"
    )

    print(
        f"Table chunks     : "
        f"{total_table_chunks}"
    )


    print(
        "\nJSONL saved to:"
    )

    print(
        OUTPUT_JSONL
    )


# ============================================================
# 15. ENTRY POINT
# ============================================================

if __name__ == "__main__":

    process_all_files()